# Лабораторная работа № 4
## *Acoustic event analysis*
по курсу Аудиоаналитика  
**направление:** Речевые технологии и машинное обучение  
**преподаватель:** Шуранов Евгений В.  
**выполнил:** Янкин Иван Ю.  
**группа:** М4121

In [ ]:
import os
import pickle
import random
from pathlib import Path

import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Audio
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
cpu_count = os.cpu_count()
num_workers = cpu_count if device == "cpu" else 0
device

## загрузка и осмотр датасета

In [ ]:
import kagglehub

base_folder = Path(kagglehub.dataset_download("ivanyankin/audionalysis-aed"))
res_folder = Path("../")
base_folder

In [ ]:
train_folder = base_folder / "train_wav" / "train"
train_csv = base_folder / "train_labels.csv"
train_pickle = res_folder / "datasets" / "train.pickle"
labels_pickle = res_folder / "datasets" / "labels.pickle"
feat_pickle = res_folder / "features" / "features.pkl"

In [ ]:
df = pd.read_csv(train_csv)
df.head(3)

### Спектрограмма

In [ ]:
sample_rate = 11025
files = [f.name for f in train_folder.iterdir() if f.is_file()]
filename = random.choice(files)

In [ ]:
filename = random.choice(files)
wav_data, sr = librosa.load(train_folder / filename, sr=None)
wav_data, sr

In [ ]:
wav_data, sr = librosa.load(train_folder / filename, sr=sample_rate)

fig, (ax_top, ax_bottom) = plt.subplots(nrows=2, ncols=1, figsize=(10, 8), sharex=True)
cmap = "viridis"

# draw linear-frequency spectrogram
hop_length = 512
wav_data_db = librosa.amplitude_to_db(np.abs(librosa.stft(wav_data, hop_length=hop_length)), ref=np.max)
img1 = librosa.display.specshow(
    wav_data_db,
    sr=sr,
    hop_length=hop_length,
    ax=ax_top,
    x_axis="time",
    y_axis="linear",
    cmap=cmap,
)
ax_top.set(title="Linear-frequency power spectrogram")
ax_top.label_outer()

# draw log-frequency spectrogram
hop_length = 1024
wav_data_db = librosa.amplitude_to_db(np.abs(librosa.stft(wav_data, hop_length=hop_length)), ref=np.max)
img2 = librosa.display.specshow(
    wav_data_db,
    sr=sr,
    hop_length=hop_length,
    ax=ax_bottom,
    x_axis="time",
    y_axis="log",
    cmap=cmap,
)
ax_bottom.set(title="Log-frequency power spectrogram")
ax_bottom.label_outer()

fig.colorbar(img1, ax=[ax_top, ax_bottom], format="%+2.f dB")
plt.show()

display(Audio(wav_data, rate=sr))

### Mel-спектрограмма

In [ ]:
# melspectrogram parameters
sample_rate = 11025
n_fft = 1024
overlap = 4
hop_length = n_fft // overlap
n_mels = 64

In [ ]:
wav_data, sr = librosa.load(train_folder / filename, sr=sample_rate)

mel_spec = librosa.feature.melspectrogram(
    y=wav_data, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels, fmax=sample_rate // 2
)

fig, (ax_top, ax_bottom) = plt.subplots(nrows=2, ncols=1, figsize=(10, 8), sharex=True)
cmap = "viridis"

img1 = librosa.display.specshow(
    mel_spec,
    sr=sr,
    fmax=sr // 2,
    ax=ax_top,
    x_axis="time",
    y_axis="mel",
    cmap=cmap,
)
ax_top.set(title="Мелспектрограмма")
ax_top.label_outer()

D = librosa.power_to_db(mel_spec, ref=np.max)
img2 = librosa.display.specshow(
    D,
    sr=sr,
    fmax=sr // 2,
    ax=ax_bottom,
    x_axis="time",
    y_axis="mel",
    cmap=cmap,
)
ax_bottom.set(title="Логарифм мелспектрограммы")
ax_bottom.label_outer()

fig.colorbar(img1, ax=[ax_top, ax_bottom], format="%+2.f dB")
plt.show()

display(Audio(wav_data, rate=sr))

## извлечение признаков

In [ ]:
def apply_pre_emphasis(y, coef=0.97):
    return np.append(y[0], y[1:] - coef * y[:-1])


def compute_log_mel(
    wav_data: np.ndarray,
    sr: int,
    n_fft: int,
    hop_length: int,
    n_mels: int,
    target_len: int = 172,
) -> np.ndarray:

    wav_data = apply_pre_emphasis(wav_data)

    wav_data = wav_data / (np.max(np.abs(wav_data)) + 1e-9)

    mel_spec = librosa.feature.melspectrogram(
        y=wav_data,
        sr=sr,
        n_fft=n_fft,
        hop_length=hop_length,
        n_mels=n_mels,
        fmax=sr // 2,
    )
    log_mel = librosa.power_to_db(mel_spec, ref=np.max)

    if log_mel.shape[1] < target_len:
        pad_width = target_len - log_mel.shape[1]
        log_mel = np.pad(log_mel, ((0, 0), (0, pad_width)), mode="constant")
    else:
        log_mel = log_mel[:, :target_len]

    return log_mel


def process_single_audio(file_name, path_to_files, sr, n_fft, hop_length, n_mels, label_id=None):
    file_path = Path(path_to_files) / file_name

    wav, current_sr = librosa.load(file_path, sr=sr, res_type="kaiser_fast")
    wav, _ = librosa.effects.trim(wav)

    log_mel_spec = compute_log_mel(wav, current_sr, n_fft, hop_length, n_mels)

    result = {
        "fname": file_name,
        "feature": log_mel_spec.astype(np.float32),
    }
    if label_id is not None:
        result["label_id"] = label_id

    return result

In [ ]:
duration_sec = 2
sample_rate = 16000
n_fft = 1024
hop_length = n_fft // 4  # 256
n_mels = 128

window_size = int((duration_sec * sample_rate) / hop_length)

In [ ]:
from joblib import Parallel, delayed

if not feat_pickle.exists():
    meta = pd.read_csv(train_csv, skiprows=1, names=["fname", "label"])
    uniq_labels = np.sort(meta["label"].unique())
    label_to_id = {label: i for i, label in enumerate(uniq_labels)}

    feats = Parallel(n_jobs=2)(
        delayed(process_single_audio)(
            file_name=row.fname,
            path_to_files=train_folder,
            sr=sample_rate,
            n_fft=n_fft,
            hop_length=hop_length,
            n_mels=n_mels,
            label_id=label_to_id[row.label],
        )
        for row in tqdm(meta.itertuples(index=False), total=len(meta))
    )

    with feat_pickle.open("wb") as f:
        pickle.dump(feats, f)